In [7]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans


**1. Data Engineering**

1.1 Data Load

In [9]:
file_name = "/KELVORA_Case_Data.xlsx"
data_frames = pd.read_excel(file_name, sheet_name=None)
hcp_m=data_frames['hcp_master']
hcp_mo_act=data_frames['hcp_monthly_activity']
patient=data_frames['patient_therapy_cohort']
enco=data_frames['economics_reference']

print(hcp_m.shape, hcp_mo_act.shape, patient.shape, enco.shape)



(620, 9) (14880, 6) (3942, 13) (12, 4)


1.2 Data Cleaning and Transformation for Selective Features

In [10]:
# Compute the percentage of HOPD and Home Infusion per HCP.
site_mix = (patient.groupby('hcp_id')['site_of_care']
            .value_counts(normalize=True)
            .unstack(fill_value=0))
site_mix = site_mix.rename(columns={
    'Hospital Outpatient (HOPD)': 'pct_hopd',
    'Home Infusion': 'pct_home_infusion',
})[['pct_hopd', 'pct_home_infusion']]

site_mix.head()




site_of_care,pct_hopd,pct_home_infusion
hcp_id,,
HCP10001,0.222222,0.444444
HCP10002,0.400000,0.400000
HCP10003,1.000000,0.000000
HCP10004,0.125000,0.750000
HCP10006,0.000000,1.000000


In [11]:
#PA rate per HCP
pa_required = patient[patient['prior_auth_required_flag'] == 1]


pa_failure = (pa_required
              .assign(failed=pa_required['prior_auth_outcome'].isin(
                  ['Abandoned - no response', 'Denied - not appealed']))
              .groupby('hcp_id')['failed']
              .mean()
              .rename('pa_failure_rate'))

pa_failure.head()


,pa_failure_rate
hcp_id,
HCP10001,0.111111
HCP10002,0.153846
HCP10003,0.500000
HCP10004,0.000000
HCP10006,1.000000


In [12]:
# Median_days_to_infusion
# Use Median over Mean for a couple of reasons (slightly better than mean)
# K mean is senesitive to extreme value
# The wait-time distribution is right-skewed (skew = 2.32):
# most patients start within ~17 days,.but some outliers have up to 121 days.
# The median reports what an actual middle patient experienced
# and isn't moved by how extreme the outliers are.
# better represents the typical patient’s wait time.

# Keep patients who actually initiated treatment

initiated = patient[patient['discontinuation_reason'] != 'Never initiated'].copy()

median_days_to_infusion = (initiated.groupby('hcp_id')['days_referral_to_first_infusion']
                    .median()
                    .rename('median_days_to_infusion'))

median_days_to_infusion.head(5)

,median_days_to_infusion
hcp_id,
HCP10001,15.5
HCP10002,14.1
HCP10003,69.6
HCP10004,11.5
HCP10007,19.8


In [14]:
# # Dose vs label ratio: actual monthly dose delivered vs. the label target, as a ratio.
# 1.0 = exactly on target, <1.0 = underdosed, >1.0 = overdosed.

# Scoped to patients who actually received infusions -- undefined for a
# never-initiated referral (no dose to evaluate).

# Get target dose and avg weight from Enco Tab
# Compute the ecno metrics
vals = dict(zip(enco['parameter'], enco['value']))
target_dose = float(vals['target_maintenance_dose_g_per_kg_month'])
avg_weight = float(vals['avg_patient_weight_kg'])



dosed = patient[patient['months_on_therapy'] > 0].copy()
infusion_per_month = 30.0 / dosed['infusion_interval_days']
monthly_grams = dosed['avg_dose_grams_per_infusion'] * infusion_per_month
dosed['dose_ratio'] = (monthly_grams / avg_weight) / target_dose

dose_vs_label = (dosed.groupby('hcp_id')['dose_ratio']
                 .mean()
                 .rename('dose_vs_label_ratio'))

dose_vs_label.head()



,dose_vs_label_ratio
hcp_id,
HCP10001,1.226296
HCP10002,1.036381
HCP10003,0.758065
HCP10004,1.161570
HCP10007,1.573389


In [ ]:
initiated.head(5)

,patient_id,hcp_id,referral_month,payer_channel,site_of_care,prior_auth_required_flag,prior_auth_outcome,days_referral_to_first_infusion,avg_dose_grams_per_infusion,infusion_interval_days,months_on_therapy,censored_flag,discontinuation_reason
0,PT200001,HCP10007,2024-07,Commercial,Physician Office / AIC,1,Approved,13.5,98.8,22.0,11.5,0,Switched to SCIg
1,PT200002,HCP10010,2024-07,Medicare Advantage,Home Infusion,1,Approved,11.0,65.1,30.0,6.4,0,Payer/access
2,PT200003,HCP10010,2024-07,Medicare Advantage,Hospital Outpatient (HOPD),1,Approved,27.4,65.0,26.0,4.2,0,Remission / taper
3,PT200004,HCP10010,2024-07,Commercial,Hospital Outpatient (HOPD),1,Approved,26.7,66.9,32.0,7.6,0,Payer/access
4,PT200005,HCP10016,2024-07,Commercial,Home Infusion,1,Approved,12.2,88.8,28.0,17.8,0,Loss of response


In [15]:
# Early Discontinuation Rate: To measure persistence behavior, it help distinguishes HCPs
# who successfully initiate patients but lose them early
# Numberator=initiated patitents who discontuned before 6 months. Denominator= patients
# whose 6-month outcome is actually evaluable.

# Evaluable cohort rules:
#   - already discontinued (any point)      -> evaluable, outcome is known
#   - ongoing AND >= 6 months observed      -> evaluable, they passed the mark
#   - ongoing AND < 6 months observed       -> EXCLUDED, outcome not yet knowable

initiated['is_evaluable'] = ((initiated['discontinuation_reason'] != 'Ongoing (censored)')| (initiated['months_on_therapy'] >= 6))

initiated['is_early_disc'] = ((initiated['discontinuation_reason'] != 'Ongoing (censored)') & (initiated['months_on_therapy'] < 6))

early_disc = (initiated[initiated['is_evaluable']]
              .groupby('hcp_id')['is_early_disc']
              .mean()
              .rename('early_discontinuation_rate'))


print('Evaluable cohort:', initiated['is_evaluable'].sum(), 'of', len(initiated), 'initiated patients')
early_disc



Evaluable cohort: 2601 of 3441 initiated patients


,early_discontinuation_rate
hcp_id,
HCP10001,0.000000
HCP10002,0.000000
HCP10003,0.000000
HCP10004,0.000000
HCP10007,0.000000
...,...
HCP10613,0.000000
HCP10614,0.166667
HCP10615,0.000000


In [16]:
# Compute KELVORA Start Share to measure HCP's Kelvora preference.

start_share = (
    hcp_mo_act.groupby('hcp_id')
    .agg(
        kelvora_starts=('new_starts_kelvora', 'sum'),
        competitor_starts=('new_starts_competitor', 'sum')
    )
)

start_share['total_starts'] = (start_share['kelvora_starts'] + start_share['competitor_starts'])

start_share['kelvora_start_share'] = (start_share['kelvora_starts']/start_share['total_starts'].replace(0, np.nan))

start_share.head()

,kelvora_starts,competitor_starts,total_starts,kelvora_start_share
hcp_id,,,,
HCP10000,0,1,1,0.000000
HCP10001,9,4,13,0.692308
HCP10002,15,6,21,0.714286
HCP10003,2,19,21,0.095238
HCP10004,8,2,10,0.800000


In [17]:
# Merge and impute
# Join everything onto hcp_master
# LEFT join matters: it guarantees every HCP survives even if they're missing
# from one of the feature tables (e.g. an HCP whose referrals never required
# a PA won't appear in pa_failure at all)

FINAL_FEATURES = [
    'median_days_to_infusion',
    'pct_hopd',
    'pct_home_infusion',
    'pa_failure_rate',
    'dose_vs_label_ratio',
    'kelvora_start_share',
    'early_discontinuation_rate',
]


feature= (hcp_m.set_index('hcp_id')
        .join(site_mix, how='left')
        .join(pa_failure, how='left')
        .join(dose_vs_label, how='left')
        .join(median_days_to_infusion, how='left')
        .join(early_disc, how='left')
        .join(start_share['kelvora_start_share'], how='left'))


# Impute missing values with the cohort median, but FLAG each one first.
# k-means cannot handle NaN at all -- it errors out -- so these must be filled.
# The _was_imputed flags keep the choice visible: you can always check which
# HCPs got a median stand-in rather than a real measured value.
for col in FINAL_FEATURES:
    n_missing = feature[col].isna().sum()
    if n_missing > 0:
        flag_col = f'{col}_was_imputed'
        feature[flag_col] = feature[col].isna().astype(int)
        feature[col] = feature[col].fillna(feature[col].median())
        print(f'{col}: imputed {n_missing} HCPs')

print('\nRemaining NaN:', feature[FINAL_FEATURES].isna().sum().sum())   # must be 0


median_days_to_infusion: imputed 92 HCPs
pct_hopd: imputed 76 HCPs
pct_home_infusion: imputed 76 HCPs
pa_failure_rate: imputed 81 HCPs
dose_vs_label_ratio: imputed 92 HCPs
kelvora_start_share: imputed 21 HCPs
early_discontinuation_rate: imputed 108 HCPs

Remaining NaN: 0


**2. K-means Modeling**

2.1 Standardize the features

In [20]:
# Scaling is mandatory, not optional, for k-means.
# k-means measures Euclidean DISTANCE between HCPs. Without scaling,
# avg_days_to_infusion (ranges ~3-120) would dominate every distance
# calculation purely because its raw numbers are bigger, while the rate
# features (all 0-1) would barely register. StandardScaler converts each
# column to mean=0, std=1, so every behavioral dimension gets equal say.
from sklearn.preprocessing import StandardScaler

X = feature[FINAL_FEATURES]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Shape:', X_scaled.shape)   # expect (620, 7)

# Sanity check: every column should now be centered at ~0 with std ~1
import pandas as pd
pd.DataFrame(X_scaled, columns=FINAL_FEATURES).describe().round(2).loc[['mean','std']]


Shape: (620, 7)


,median_days_to_infusion,pct_hopd,pct_home_infusion,pa_failure_rate,dose_vs_label_ratio,kelvora_start_share,early_discontinuation_rate
mean,-0.0,0.0,-0.0,0.0,-0.0,-0.0,-0.0
std,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [19]:
# Which features still have NaN?
print(feature[FINAL_FEATURES].isna().sum())
print()
print('Total rows with at least one NaN:', feature[FINAL_FEATURES].isna().any(axis=1).sum())

median_days_to_infusion       0
pct_hopd                      0
pct_home_infusion             0
pa_failure_rate               0
dose_vs_label_ratio           0
kelvora_start_share           0
early_discontinuation_rate    0
dtype: int64

Total rows with at least one NaN: 0


2.2 Determine cluster number K (silhouette scan with a minimum cluster size floor)

In [21]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Scan k = 3 through 7. Two criteria decide the winner:
#   1. SILHOUETTE SCORE -- measures how well-separated the clusters are.
#      For each HCP: compare distance to their own cluster vs. the nearest
#      other cluster. Ranges -1 to +1; higher = cleaner separation.
#   2. MINIMUM CLUSTER SIZE FLOOR (5% of the cohort, ~31 HCPs) -- a
#      mathematically tight 10-HCP cluster can score well on silhouette but
#      is commercially useless; you can't build a $200K program around it.
#      This floor is a judgment call, not a statistical requirement.
#
# n_init=10 runs each k with 10 different random starting points and keeps
# the best -- k-means is sensitive to initialization and can land in a bad
# local optimum on a single run.


results = []
n = X_scaled.shape[0]

for k in range(3, 8):
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X_scaled)
    sizes = np.bincount(labels)
    results.append({
        'k': k,
        'silhouette': silhouette_score(X_scaled, labels),
        'min_cluster_pct': sizes.min() / n,
        'passes_5pct_floor': sizes.min() / n >= 0.05,
    })

k_search = pd.DataFrame(results)
print(k_search.round(3))

# Pick the best-scoring k among those that clear the floor
eligible = k_search[k_search['passes_5pct_floor']]
best_k = int(eligible.sort_values('silhouette', ascending=False).iloc[0]['k'])
print(f'\nChosen k = {best_k}')

   k  silhouette  min_cluster_pct  passes_5pct_floor
0  3       0.239            0.161               True
1  4       0.226            0.105               True
2  5       0.238            0.066               True
3  6       0.262            0.048              False
4  7       0.229            0.052               True

Chosen k = 3


2.3 Profile the segments with K number

In [22]:
# Fit the final model at the chosen k and attach segment labels back onto
# the ORIGINAL (unscaled) feature table -- so the profile below reads in
# real, interpretable units (days, ratios, shares) rather than z-scores.
km_final = KMeans(n_clusters=best_k, n_init=10, random_state=42)
feature['segment'] = km_final.fit_predict(X_scaled)

print('Silhouette:', round(silhouette_score(X_scaled, feature['segment']), 3))
print('\nSegment sizes:')
print(feature['segment'].value_counts().sort_index())


# Profile: mean of every clustering feature by segment. This is the table
# that turns cluster numbers 0/1/2 into a commercial story.
profile = feature.groupby('segment')[FINAL_FEATURES].mean().round(3)
profile['n_hcps'] = feature.groupby('segment').size()

# Context columns -- NOT used to build the clusters, only to describe them
# after the fact (decile and volume were deliberately excluded from
# clustering, but they're useful for the narrative).
profile['modal_practice_setting'] = (feature.groupby('segment')['practice_setting']
                                     .agg(lambda s: s.value_counts().idxmax()))
profile['mean_decile'] = feature.groupby('segment')['syndicated_patient_decile'].mean().round(1)

profile.T

Silhouette: 0.239

Segment sizes:
segment
0    100
1    234
2    286
Name: count, dtype: int64


segment,0,1,2
median_days_to_infusion,39.809,19.701,17.041
pct_hopd,0.698,0.293,0.22
pct_home_infusion,0.066,0.273,0.525
pa_failure_rate,0.377,0.103,0.089
dose_vs_label_ratio,1.071,0.784,1.121
kelvora_start_share,0.32,0.318,0.62
early_discontinuation_rate,0.006,0.218,0.025
n_hcps,100,234,286
modal_practice_setting,Hospital-Affiliated Neurology,Hospital-Affiliated Neurology,Community Neurology
mean_decile,7.0,5.2,5.2


In [ ]:
print('feature index name:', feature.index.name)
print('hcp_id in feature.columns?', 'hcp_id' in feature.columns)
print('hcp_id in patient.columns?', 'hcp_id' in patient.columns)
print()
print('feature columns:', feature.columns.tolist())

feature index name: hcp_id
hcp_id in feature.columns? False
hcp_id in patient.columns? True

feature columns: ['primary_specialty', 'practice_setting', 'region', 'territory_id', 'years_in_practice', 'idn_affiliated_flag', 'onsite_infusion_suite_flag', 'syndicated_patient_decile', 'pct_hopd', 'pct_home_infusion', 'pa_failure_rate', 'dose_vs_label_ratio', 'median_days_to_infusion', 'early_discontinuation_rate', 'kelvora_start_share', 'median_days_to_infusion_was_imputed', 'pct_hopd_was_imputed', 'pct_home_infusion_was_imputed', 'pa_failure_rate_was_imputed', 'dose_vs_label_ratio_was_imputed', 'kelvora_start_share_was_imputed', 'early_discontinuation_rate_was_imputed', 'segment']


**3. Size the opportunity per segment**

In [23]:
#load those enco metrics from enco tab

nr_per_gram = float(vals['net_revenue_per_gram_usd'])
margin_pct = float(vals['contribution_margin_pct'])
pa_recovery_rate = float(vals['assumed_pa_recovery_rate'])


annual_grams = target_dose * avg_weight * 12
annual_contribution = annual_grams * nr_per_gram * margin_pct
monthly_contribution = annual_contribution / 12


#assumed recovery rate; lower than PA rate
#lower than PA recovery rate is because
#changing dosing behavior is harrder than refefilling a form
RETENTION_RECOVERY_RATE = 0.25

# a recovered patient is valued at 12 months of therapy
INCREMENTAL_MONTHS = 12


if feature.index.name == 'hcp_id':
    feature = feature.reset_index()
# Attach segment labels back onto the patient-level table.
# Grain shift: clustering happened at the HCP level (620 rows), but dollars
# come from patients (3,942 referral rows) -- this merge bridges the two.
patient_seg = patient.merge(feature[['hcp_id', 'segment']], on='hcp_id', how='left')





# --- Mechanism A: PA failure (never converted) ---
# Covers abandoned AND denied, matching assumed_pa_recovery_rate's own definition

patient_seg['is_pa_failure'] = patient_seg['prior_auth_outcome'].isin(
    ['Abandoned - no response', 'Denied - not appealed'])



# --- Mechanism B: early discontinuation (<6mo), dosing-sensitive reasons only ---
# Loss of response and tolerability are plausibly dosing/monitoring-driven.
# Excluded: "Switched to SCIg" (competitive, not addressable by these levers),
# "Payer/access" (different root cause), "Remission / taper" (good outcome).
patient_seg['is_early_disc'] = (
    patient_seg['discontinuation_reason'].isin(['Loss of response', 'Tolerability'])
    & (patient_seg['months_on_therapy'] < 6)
)

rows = []
for seg, g in patient_seg.groupby('segment'):
    n_hcps = feature.loc[feature['segment'] == seg, 'hcp_id'].nunique()
    n_pa_required = (g['prior_auth_required_flag'] == 1).sum()
    n_initiated = (g['discontinuation_reason'] != 'Never initiated').sum()

    pa_count = g['is_pa_failure'].sum()
    pa_recoverable = pa_count * pa_recovery_rate
    pa_value = pa_recoverable * INCREMENTAL_MONTHS * monthly_contribution

    ed_count = g['is_early_disc'].sum()
    ed_recoverable = ed_count * RETENTION_RECOVERY_RATE
    ed_value = ed_recoverable * INCREMENTAL_MONTHS * monthly_contribution

    rows.append({
        'segment': seg,
        'n_hcps': n_hcps,
        'pa_failed': pa_count,
        'pa_failure_rate_seg': round(pa_count / n_pa_required, 3) if n_pa_required else np.nan,
        'pa_recoverable_patients': round(pa_recoverable, 1),
        'pa_value_usd': round(pa_value),
        'early_disc': ed_count,
        'early_disc_rate_seg': round(ed_count / n_initiated, 3) if n_initiated else np.nan,
        'retention_recoverable_patients': round(ed_recoverable, 1),
        'retention_value_usd': round(ed_value),
        'total_opportunity_usd': round(pa_value + ed_value),
    })

opportunity = pd.DataFrame(rows).sort_values('total_opportunity_usd', ascending=False)
opportunity

### At this point, export or paste the opporunity table
### into excel for FY27 funding recommendation



,segment,n_hcps,pa_failed,pa_failure_rate_seg,pa_recoverable_patients,pa_value_usd,early_disc,early_disc_rate_seg,retention_recoverable_patients,retention_value_usd,total_opportunity_usd
2,2,286,214,0.091,96.3,3241319,18,0.008,4.5,151464,3392783
1,1,234,122,0.138,54.9,1847855,87,0.104,21.8,732074,2579929
0,0,100,165,0.394,74.2,2499148,2,0.007,0.5,16829,2515977


**Appendix --- Create the Master Table and save into CSV**

In [24]:
# =====================================================================
# Kelvora_HCP_Seg_Master — one row per HCP
# hcp_master attributes + rolled-up activity + patient aggregates
# + the 7 behavioral features + named segment
# =====================================================================

# --- 1. Name the clusters by their defining behavior -----------------
# k-means label NUMBERS are not stable across runs, so never hardcode
# "cluster 0 = Access-Blocked". Identify each group by the feature it
# leads on, so the naming survives a re-run.
seg_profile = feature.groupby('segment')[FINAL_FEATURES].mean()

segment_names = {
    seg_profile['pa_failure_rate'].idxmax():            'Access-Blocked',
    seg_profile['early_discontinuation_rate'].idxmax(): 'Underdosed & Dropping Off',
    seg_profile['kelvora_start_share'].idxmax():        'KELVORA Loyalists',
}

# Guard: if two of those resolve to the same cluster, the naming rule
# has broken and we want to know rather than silently mislabel.
assert len(segment_names) == 3, f'Segment naming collided: {segment_names}'

feature['segment_name'] = feature['segment'].map(segment_names)
print(feature['segment_name'].value_counts())




# --- 2. Roll up monthly activity to HCP level -----------------------
activity_roll = (
    hcp_mo_act.groupby('hcp_id')
    .agg(
        total_kelvora_starts=('new_starts_kelvora', 'sum'),
        total_competitor_starts=('new_starts_competitor', 'sum'),
        total_grams_shipped=('kelvora_grams_shipped', 'sum'),
        avg_patients_on_therapy=('patients_on_kelvora_eom', 'mean'),
        peak_patients_on_therapy=('patients_on_kelvora_eom', 'max'),
        months_observed=('month', 'nunique'),
    )
)
# --- 3. Roll up the patient journey to HCP level ---------------------
# Flags first, then one groupby — cheaper and easier to read than
# several passes over the same table.
pat = patient.copy()
pat['is_never_initiated'] = pat['discontinuation_reason'].eq('Never initiated')
pat['is_pa_required']     = pat['prior_auth_required_flag'].eq(1)
pat['is_pa_failed']       = pat['prior_auth_outcome'].isin(
                                ['Abandoned - no response', 'Denied - not appealed'])
pat['is_ongoing']         = pat['discontinuation_reason'].eq('Ongoing (censored)')
pat['is_scig_switch']     = pat['discontinuation_reason'].eq('Switched to SCIg')

patient_roll = (
    pat.groupby('hcp_id')
    .agg(
        n_referrals=('patient_id', 'count'),
        n_never_initiated=('is_never_initiated', 'sum'),
        n_pa_required=('is_pa_required', 'sum'),
        n_pa_failed=('is_pa_failed', 'sum'),
        n_still_on_therapy=('is_ongoing', 'sum'),
        n_switched_to_scig=('is_scig_switch', 'sum'),
        median_months_on_therapy=('months_on_therapy', 'median'),
    )
)
patient_roll['n_initiated'] = patient_roll['n_referrals'] - patient_roll['n_never_initiated']


# --- 4. Assemble -----------------------------------------------------
# hcp_master is the spine: LEFT joins so all 620 survive even where a
# prescriber has no referrals or no recorded activity.
CONTEXT_COLS = ['primary_specialty', 'practice_setting', 'region', 'territory_id',
                'years_in_practice', 'idn_affiliated_flag',
                'onsite_infusion_suite_flag', 'syndicated_patient_decile']

Kelvora_HCP_Seg_Master = (
    hcp_m.set_index('hcp_id')[CONTEXT_COLS]
    .join(feature.set_index('hcp_id')[['segment', 'segment_name'] + FINAL_FEATURES], how='left')
    .join(activity_roll, how='left')
    .join(patient_roll, how='left')
    .reset_index()
)

# Counts are genuinely zero where a prescriber has no records, not unknown
count_cols = [c for c in Kelvora_HCP_Seg_Master.columns if c.startswith('n_') or c.startswith('total_')]
Kelvora_HCP_Seg_Master[count_cols] = Kelvora_HCP_Seg_Master[count_cols].fillna(0).astype(int)

# Order columns so the table reads: who → which segment → behavior → volume
Kelvora_HCP_Seg_Master = Kelvora_HCP_Seg_Master[
    ['hcp_id', 'segment_name', 'segment'] + CONTEXT_COLS + FINAL_FEATURES +
    list(patient_roll.columns) + list(activity_roll.columns)
]

print('rows:', len(Kelvora_HCP_Seg_Master), '| expect 620')
print('unassigned segments:', Kelvora_HCP_Seg_Master['segment_name'].isna().sum())
Kelvora_HCP_Seg_Master.head()




segment_name
KELVORA Loyalists            286
Underdosed & Dropping Off    234
Access-Blocked               100
Name: count, dtype: int64
rows: 620 | expect 620
unassigned segments: 0


,hcp_id,segment_name,segment,primary_specialty,practice_setting,region,territory_id,years_in_practice,idn_affiliated_flag,onsite_infusion_suite_flag,...,n_still_on_therapy,n_switched_to_scig,median_months_on_therapy,n_initiated,total_kelvora_starts,total_competitor_starts,total_grams_shipped,avg_patients_on_therapy,peak_patients_on_therapy,months_observed
0,HCP10000,Underdosed & Dropping Off,1,Neurology,Community Neurology,Central,T038,5,0,1,...,0,0,NaN,0,0,1,0,0.000000,0,24
1,HCP10001,KELVORA Loyalists,2,Neurology,Community Neurology,Northeast,T008,24,0,1,...,6,0,13.30,8,9,4,10148,4.041667,7,24
2,HCP10002,KELVORA Loyalists,2,Neurology,Community Neurology,West,T028,15,0,0,...,9,2,6.00,13,15,6,10526,5.208333,9,24
3,HCP10003,Access-Blocked,0,Neurology,Community Neurology,Central,T015,23,0,0,...,0,1,4.50,1,2,19,539,0.375000,1,24
4,HCP10004,KELVORA Loyalists,2,Neuromuscular Neurology,Academic Medical Center,Southeast,T014,14,1,1,...,6,0,8.55,8,8,2,8059,3.625000,7,24


Export into CSV

In [25]:
# --- 6. Export -------------------------------------------------------
Kelvora_HCP_Seg_Master.to_excel('Kelvora_HCP_Seg_Master.xlsx', index=False)

from google.colab import files
files.download('Kelvora_HCP_Seg_Master.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>